# Examining the Data
Both the clayton digest and sales data were already compiled, so we don't actually need to do any of the compiling work here.
I will turn the clayton sales into a csv though, and verify for lack of duplicates.

**Tax Digest**: clayton_digest_v2.csv  
**Sales Data**: CLAYTON_Sales Data 2010through2023.xlsx

In [33]:
import os
import pandas as pd

In [39]:
SALES_PATH = "/Users/tpeng/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Housing and Urban Policy (HUP) Lab - Documents/Data Files/Raw Data/ATLSales/FINAL/CLAYTON_SALES_FINAL.csv"
#DIGEST_PATH = "/Users/tpeng/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Housing and Urban Policy (HUP) Lab - Documents/Data Files/Final Datasets/Clayton/clayton_digest_v2.csv"
DIGEST_PATH = "../../data/clayton/clayton_digest_v2.csv"
OUT_PATH = "../../data/clayton/out"

if os.path.exists(SALES_PATH):
    print(f"Sales file present at {SALES_PATH}")
else:
    print(f"Sales file NOT PRESENT, {SALES_PATH} not found.")

if os.path.exists(DIGEST_PATH):
    print(f"Digest file present at {DIGEST_PATH}")
else:
    print(f"Digest file NOT PRESENT, {DIGEST_PATH} not found.")

Sales file present at /Users/tpeng/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Housing and Urban Policy (HUP) Lab - Documents/Data Files/Raw Data/ATLSales/FINAL/CLAYTON_SALES_FINAL.csv
Digest file present at ../../data/clayton/clayton_digest_v2.csv


In [40]:
sales_df = pd.read_csv(SALES_PATH, parse_dates=["SALEDT"])

In [41]:
sales_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157647 entries, 0 to 157646
Data columns (total 12 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   PARID     157647 non-null  object        
 1   SALEDT    157647 non-null  datetime64[ns]
 2   PRICE     157647 non-null  int64         
 3   NEWYR     124815 non-null  float64       
 4   BOOK      157647 non-null  object        
 5   PAGE      157647 non-null  object        
 6   OLDOWN    152274 non-null  object        
 7   OWN1      157643 non-null  object        
 8   SALETYPE  157187 non-null  object        
 9   SALEVAL   157624 non-null  object        
 10  MKTVALID  157490 non-null  object        
 11  INSTRTYP  157625 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(9)
memory usage: 14.4+ MB


In [42]:
digest_df = pd.read_csv(DIGEST_PATH, low_memory=False)

In [14]:
digest_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999809 entries, 0 to 999808
Columns: 162 entries, tax_year to is_nonprofit
dtypes: bool(3), float64(86), int64(9), object(64)
memory usage: 1.2+ GB


Verify whether that tax year and pin pairs are unique (and therefore pin is a valid unique parcel id).

In [43]:
digest_df.groupby(by=["pin", "tax_year"]).count().max().max()

1

# Merging
We have to create a sale year column so we can merge with the tax digest.

In [45]:
digest_df.describe()

,tax_year,taxdistrict,streetno,buildinglocation,locationposition,zipcode,zone2,zone3,businesscode2,saledate,...,buscode2desc,subdname,apartments,commission,improvemnt,zoningurl,own_corp_flag,is_non_owner_occupied,rental_flag,is_nonprofit
count,999809.000000,999809.000000,9.513180e+05,0.0,0.0,999805.000000,0.0,0.0,0.0,9.998090e+05,...,0.0,0.0,0.0,984016.000000,6.052100e+05,0.0,999809.000000,999809.000000,999809.000000,999809.000000
mean,2017.017220,7.401797,4.657760e+03,NaN,NaN,30265.685161,NaN,NaN,NaN,2.010938e+07,...,NaN,NaN,NaN,2.514408,2.048343e+05,NaN,0.246902,0.124751,0.445045,0.001308
std,3.165938,1.513054,6.161141e+04,NaN,NaN,32.155733,NaN,NaN,NaN,1.066045e+05,...,NaN,NaN,NaN,1.134293,8.982010e+06,NaN,0.431210,0.330436,0.496971,0.036146
min,2012.000000,1.000000,0.000000e+00,NaN,NaN,30215.000000,NaN,NaN,NaN,1.939020e+07,...,NaN,NaN,NaN,1.000000,-1.110000e+04,NaN,0.000000,0.000000,0.000000,0.000000
25%,2014.000000,8.000000,1.198000e+03,NaN,NaN,30236.000000,NaN,NaN,NaN,2.004092e+07,...,NaN,NaN,NaN,1.000000,4.517800e+04,NaN,0.000000,0.000000,0.000000,0.000000
50%,2017.000000,8.000000,4.164000e+03,NaN,NaN,30260.000000,NaN,NaN,NaN,2.014090e+07,...,NaN,NaN,NaN,3.000000,8.100000e+04,NaN,0.000000,0.000000,0.000000,0.000000
75%,2020.000000,8.000000,7.246000e+03,NaN,NaN,30294.000000,NaN,NaN,NaN,2.019111e+07,...,NaN,NaN,NaN,4.000000,1.277760e+05,NaN,0.000000,0.000000,1.000000,0.000000
max,2022.000000,9.000000,3.000258e+07,NaN,NaN,30354.000000,NaN,NaN,NaN,2.049061e+07,...,NaN,NaN,NaN,4.000000,2.013750e+09,NaN,1.000000,1.000000,1.000000,1.000000


In [44]:
sales_df["SALE_YR"] = sales_df["SALEDT"].dt.year

In [57]:
sales_eligible_df = sales_df[(sales_df['SALE_YR'] >= 2012) & (sales_df['SALE_YR'] <= 2022) & 
                             (~sales_df['PARID'].str.startswith(("UTILITY", "AIRPORT", "UITILITY")))]
sales_tax_merged = pd.merge(sales_eligible_df, digest_df, how="left", left_on=["PARID", "SALE_YR"], right_on=["pin", "tax_year"], suffixes=("_sales", ""))

In [58]:
total_rows = sales_tax_merged.shape[0]
merged_rows = sales_tax_merged["pin"].notna().sum()

print(f"{merged_rows / total_rows} succesfully merged.\n {merged_rows} out of {total_rows}")

0.9701795280742649 succesfully merged.
 118619 out of 122265


In [59]:
sales_tax_merged.to_csv(os.path.join(OUT_PATH, "CLAYTON_SALES_TAX_DIGEST_FINAL.csv"), index=False)

# Investigating Unmerged Rows

In [60]:
unmerged_df = sales_tax_merged[sales_tax_merged["pin"].isna()]
merged_df = sales_tax_merged[sales_tax_merged["pin"].notna()]

Check if any of the numeric columns have odd properties when comparing between merged and unmerged rows.

In [61]:
unmerged_df.describe()

,SALEDT,PRICE,NEWYR,SALE_YR,tax_year,taxdistrict,streetno,buildinglocation,locationposition,zipcode,...,buscode2desc,subdname,apartments,commission,improvemnt,zoningurl,own_corp_flag,is_non_owner_occupied,rental_flag,is_nonprofit
count,3646,3.646000e+03,2841.000000,3646.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,2018-08-30 16:58:58.782226944,6.322804e+05,2019.221753,2018.057597,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,2012-01-18 00:00:00,0.000000e+00,2013.000000,2012.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,2017-04-28 00:00:00,0.000000e+00,2018.000000,2017.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,2018-10-04 00:00:00,1.000000e+00,2019.000000,2018.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2020-12-18 00:00:00,9.000000e+05,2020.000000,2020.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,2022-12-30 00:00:00,9.090590e+07,2024.000000,2022.000000,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,1.938325e+06,2.388816,2.624978,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
merged_df.describe()

,SALEDT,PRICE,NEWYR,SALE_YR,tax_year,taxdistrict,streetno,buildinglocation,locationposition,zipcode,...,buscode2desc,subdname,apartments,commission,improvemnt,zoningurl,own_corp_flag,is_non_owner_occupied,rental_flag,is_nonprofit
count,118619,1.186190e+05,94074.000000,118619.000000,118619.000000,118619.000000,1.150770e+05,0.0,0.0,118619.000000,...,0.0,0.0,0.0,115028.000000,7.241100e+04,0.0,118619.000000,118619.000000,118619.000000,118619.000000
mean,2017-03-28 12:30:30.060951296,4.858117e+05,2016.621978,2016.725010,2016.725010,7.464782,5.117528e+03,NaN,NaN,30264.828021,...,NaN,NaN,NaN,2.544337,1.269586e+05,NaN,0.402828,0.217166,0.690539,0.000919
min,2012-01-01 00:00:00,0.000000e+00,2010.000000,2012.000000,2012.000000,1.000000,0.000000e+00,NaN,NaN,30215.000000,...,NaN,NaN,NaN,1.000000,0.000000e+00,NaN,0.000000,0.000000,0.000000,0.000000
25%,2014-05-08 00:00:00,0.000000e+00,2014.000000,2014.000000,2014.000000,8.000000,1.235000e+03,NaN,NaN,30236.000000,...,NaN,NaN,NaN,2.000000,2.806000e+04,NaN,0.000000,0.000000,0.000000,0.000000
50%,2017-03-16 00:00:00,6.000000e+04,2016.000000,2017.000000,2017.000000,8.000000,4.135000e+03,NaN,NaN,30260.000000,...,NaN,NaN,NaN,3.000000,6.738900e+04,NaN,0.000000,0.000000,1.000000,0.000000
75%,2019-12-20 00:00:00,1.600000e+05,2019.000000,2019.000000,2019.000000,8.000000,7.351000e+03,NaN,NaN,30294.000000,...,NaN,NaN,NaN,3.000000,1.139515e+05,NaN,1.000000,0.000000,1.000000,0.000000
max,2022-12-30 00:00:00,1.059200e+08,2024.000000,2022.000000,2022.000000,9.000000,3.000258e+07,NaN,NaN,30354.000000,...,NaN,NaN,NaN,4.000000,3.953800e+07,NaN,1.000000,1.000000,1.000000,1.000000
std,NaN,2.573782e+06,2.633920,3.201766,3.201766,1.494829,1.251092e+05,NaN,NaN,33.324771,...,NaN,NaN,NaN,1.111425,7.773966e+05,NaN,0.490469,0.412318,0.462274,0.030300


## Determine which ParcelIDs are present and which are not
We want to check to see if the lack of a match is due only to the lack of a match in ParcelID, or specifically a lack of match (year, ParcelID) pairs.

In [63]:
unmerged_pids = pd.Series(unmerged_df["PARID"].unique())
print(f"{unmerged_pids.size} unmerged unique parcelIDs")

2790 unmerged unique parcelIDs


In [64]:
not_present = unmerged_pids[~unmerged_pids.isin(digest_df["pin"])]
present = unmerged_pids[unmerged_pids.isin(digest_df["pin"])]
print(f"{not_present.size} unmerged parcelIDs not present in digest")

595 unmerged parcelIDs not present in digest


In [65]:
not_present.str.split(" ").str[0].unique()

array(['13007', '04243A', '13073B', '13070B', '05115', '12177B', '06160D',
       '12210', '04207', '13185D', '13183D', '06100', '06164C', '13050A',
       '06132C', '13240D', '06133A', '13078B', '12073C', '13151D',
       '12015C', '12151C', '06159C', '06164B', '12074D', '06157B',
       '06157D', '06038', '06094', '12010B', '13071C', '05239C', '06131D',
       '05248A', '13248C', '12204', '13151C', '13233B', '06132A',
       '13242B', '13078C', '12083A', '13210C', '06164D', '12108C',
       '06130', '13167C', '05240B', '12075D'], dtype=object)

In [75]:
present

0         12176A B015
1         13134A B017
2         13215C B008
3         13215C B009
4         13175B E014
            ...      
2446      06159C F055
2448      06159C F050
2449      06159C F049
2450      06159C F052
2456    05109 110013Y
Length: 2195, dtype: object

In [67]:
unmerged_not_present_df = unmerged_df[unmerged_df["PARID"].isin(not_present)]
unmerged_present_df = unmerged_df[unmerged_df["PARID"].isin(present)]

In [68]:
unmerged_not_present_df.shape

(671, 175)

In [69]:
unmerged_present_df.shape

(2975, 175)

In [70]:
unmerged_present_df["year_before"] = unmerged_present_df["SALE_YR"] - 1
unmerged_present_df["year_2before"] = unmerged_present_df["SALE_YR"] - 2
unmerged_present_df["year_after"] = unmerged_present_df["SALE_YR"] + 1

unmerged_year_status_df = pd.merge(unmerged_present_df, digest_df[["pin", "tax_year"]], left_on=["PARID", "year_before"], right_on=["pin", "tax_year"], how="left", suffixes=("", "_before"))
unmerged_year_status_df = pd.merge(unmerged_year_status_df, digest_df[["pin", "tax_year"]], left_on=["PARID", "year_2before"], right_on=["pin", "tax_year"], how="left", suffixes=("", "_2before"))
unmerged_year_status_df = pd.merge(unmerged_year_status_df, digest_df[["pin", "tax_year"]], left_on=["PARID", "year_after"], right_on=["pin", "tax_year"], how="left", suffixes=("", "_after"))

/var/folders/bb/g7vlcgfn14ld8_w41331g0dw0000gn/T/ipykernel_507/1056661330.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unmerged_present_df["year_before"] = unmerged_present_df["SALE_YR"] - 1
/var/folders/bb/g7vlcgfn14ld8_w41331g0dw0000gn/T/ipykernel_507/1056661330.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unmerged_present_df["year_2before"] = unmerged_present_df["SALE_YR"] - 2
/var/folders/bb/g7vlcgfn14ld8_w41331g0dw0000gn/T/ipykernel_507/1056661330.py:3: SettingWithCopyWarning: 
A value is

In [73]:
before = (~unmerged_year_status_df.pin_before.isna()).sum()
after = (~unmerged_year_status_df.pin_after.isna()).sum()
both = ((~unmerged_year_status_df.pin_before.isna()) & (~unmerged_year_status_df.pin_after.isna())).sum()
neither = ((unmerged_year_status_df.pin_before.isna()) & (unmerged_year_status_df.pin_after.isna())).sum()

print(f"{before}/{unmerged_present_df.shape[0]} parcels existed before sale\n{after}/{unmerged_present_df.shape[0]} parcels existed after \n{both}/{unmerged_present_df.shape[0]} parcels existed both before and after sale.\n" + 
      f"{neither}/{unmerged_present_df.shape[0]} parcels existed neither before nor after.")

10/2975 parcels existed before sale
1576/2975 parcels existed after 
2/2975 parcels existed both before and after sale.
1391/2975 parcels existed neither before nor after.


In [74]:
unmerged_not_present_df.to_csv(os.path.join(OUT_PATH, "CLAYTON_UNMATCHED_NOT_PRESENT.csv"), index=False)
unmerged_year_status_df.to_csv(os.path.join(OUT_PATH, "CLAYTON_UNMATCHED_PRESENT.csv"), index=False)